# Highlands Ranch, CO LVT Shift Notebook

This notebook follows the same overall workflow as `examples/greeley.ipynb`, but uses Douglas County parcel geometry + assessor flat files to build a Highlands Ranch-specific dataset.


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from shapely.geometry import Polygon

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "lvt_utils.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from lvt_utils import calculate_category_tax_summary, model_split_rate_tax, print_category_tax_summary
from policy_analysis import analyze_land_by_improvement_share, analyze_parking_lots, analyze_vacant_land, print_parking_analysis_summary, print_vacant_land_summary
from census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from viz import create_quintile_summary

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)


In [ ]:
data_dir = REPO_ROOT / "examples" / "data" / "highlands_ranch"
data_dir.mkdir(parents=True, exist_ok=True)

parcel_query_url = "https://apps.douglas.co.us/geopendata/rest/services/Parcels/Parcel/MapServer/0/query"
fallback_parcel_query_url = "https://services5.arcgis.com/JlofwxJO3RD8jjJH/arcgis/rest/services/Map_WFL1/FeatureServer/0/query"
zoning_query_url = "https://services5.arcgis.com/JlofwxJO3RD8jjJH/arcgis/rest/services/Map_WFL1/FeatureServer/13/query"

flat_file_base = "https://apps.douglas.co.us/realware/datadownloads/"
location_url = flat_file_base + "Property_Location.txt"
ownership_url = flat_file_base + "Property_Ownership.txt"
subdivision_url = flat_file_base + "Property_Subdivision.txt"
value_url = flat_file_base + "Property_Value.txt"
sales_url = flat_file_base + "Property_Sales.txt"

hr_city = "HIGHLANDS RANCH"


## Step 1: Load assessor flat files and filter to Highlands Ranch


In [ ]:
def load_pipe_txt(url, cache_path):
    if cache_path.exists():
        print(f"Loading cache: {cache_path.name}")
        return pd.read_parquet(cache_path)
    print(f"Downloading: {url}")
    df = pd.read_csv(url, sep="|", dtype=str, low_memory=False)
    df.to_parquet(cache_path, index=False)
    return df

location_df = load_pipe_txt(location_url, data_dir / "property_location_20260401.parquet")
ownership_df = load_pipe_txt(ownership_url, data_dir / "property_ownership_20260401.parquet")
subdivision_df = load_pipe_txt(subdivision_url, data_dir / "property_subdivision_20260401.parquet")
value_df = load_pipe_txt(value_url, data_dir / "property_value_20260401.parquet")
sales_df = load_pipe_txt(sales_url, data_dir / "property_sales_20260401.parquet")

location_hr = location_df[location_df["City_Name"].fillna("").str.upper().eq(hr_city)].copy()
print(f"HR rows from location table: {len(location_hr):,}")


In [ ]:
value_df["Actual_Value"] = pd.to_numeric(value_df.get("Actual_Value"), errors="coerce").fillna(0)
value_df["Assessed_Value"] = pd.to_numeric(value_df.get("Assessed_Value"), errors="coerce").fillna(0)

value_segments = (
    value_df.groupby(["Account_No", "Valuation_Type_Code"], dropna=False)
    .agg(seg_actual=("Actual_Value", "sum"), seg_assessed=("Assessed_Value", "sum"))
    .reset_index()
)
land_seg = value_segments[value_segments["Valuation_Type_Code"] == "L"].rename(columns={"seg_actual": "LANDACT", "seg_assessed": "LANDASD"})[["Account_No", "LANDACT", "LANDASD"]]
imp_seg = value_segments[value_segments["Valuation_Type_Code"] == "I"].rename(columns={"seg_actual": "IMPACT", "seg_assessed": "IMPASD"})[["Account_No", "IMPACT", "IMPASD"]]

totals = value_df.groupby("Account_No", dropna=False).agg(
    TOTALACT=("Actual_Value", "sum"),
    TOTALASD=("Assessed_Value", "sum"),
    Valuation_Description=("Valuation_Description", "first"),
    Valuation_Class_Code=("Valuation_Class_Code", "first"),
    Account_Type_Code_Value=("Account_Type_Code", "first"),
).reset_index()

account_values = totals.merge(land_seg, on="Account_No", how="left").merge(imp_seg, on="Account_No", how="left")
for col in ["LANDACT", "LANDASD", "IMPACT", "IMPASD"]:
    account_values[col] = pd.to_numeric(account_values[col], errors="coerce").fillna(0)


In [ ]:
owners = ownership_df.drop_duplicates(subset=["Account_No"])
subs = subdivision_df.drop_duplicates(subset=["Account_No"])

sales_df["Sale_Date"] = pd.to_datetime(sales_df.get("Sale_Date"), errors="coerce")
sales_df["Sale_Price"] = pd.to_numeric(sales_df.get("Sale_Price"), errors="coerce")
latest_sales = sales_df.sort_values(["Account_No", "Sale_Date"]).drop_duplicates("Account_No", keep="last")

attrs = (
    location_hr
    .merge(account_values, on="Account_No", how="left")
    .merge(owners[["Account_No", "Owner_Name"]], on="Account_No", how="left")
    .merge(subs[["Account_No", "Subdivision_Name"]], on="Account_No", how="left")
    .merge(latest_sales[["Account_No", "Sale_Date", "Sale_Price"]], on="Account_No", how="left")
)

attrs["ACCTTYPE"] = attrs["Account_Type_Code"].fillna(attrs.get("Account_Type_Code_Value"))
attrs["PARCEL"] = attrs["State_Parcel_No"]
attrs["ACCOUNTNO"] = attrs["Account_No"]
attrs["NAME"] = attrs["Owner_Name"]
attrs["LOCCITY"] = attrs["City_Name"]
attrs["SALEP"] = attrs["Sale_Price"].fillna(0)
attrs["SALEDT"] = attrs["Sale_Date"]
attrs["ASSRCODE"] = attrs["Valuation_Class_Code"]
attrs["GIS_Acres"] = pd.to_numeric(attrs.get("Total_Net_Acres"), errors="coerce")


## Step 2: Load geometry and merge


In [ ]:
def esri_poly_to_shapely(geom):
    rings = geom.get("rings", []) if isinstance(geom, dict) else []
    if not rings:
        return None
    return Polygon(rings[0], holes=rings[1:] if len(rings) > 1 else None)


def fetch_geometries(spns, query_url):
    session = requests.Session()
    all_rows = []
    uniq = [x for x in pd.Series(spns).dropna().astype(str).unique().tolist() if x]
    for i in range(0, len(uniq), 250):
        chunk = uniq[i:i+250]
        quoted = ",".join(["'" + x.replace("'", "''") + "'" for x in chunk])
        where = f"PARCEL_SPN IN ({quoted})"
        r = session.get(query_url, params={
            "f": "json", "where": where,
            "outFields": "OBJECTID,PARCEL_SPN,DEEDED_AREA,CALC_AREA,LEGAL_DESCR,PARCEL_NAME,BLOCK_NO,Shape__Area,Shape__Length",
            "returnGeometry": "true", "outSR": 4326,
        }, timeout=180)
        r.raise_for_status()
        all_rows.extend(r.json().get("features", []))
        print(f"chunk {min(i+250, len(uniq)):,}/{len(uniq):,}")
    rows=[]
    for f in all_rows:
        a=f.get("attributes",{})
        g=esri_poly_to_shapely(f.get("geometry",{}))
        if g is not None:
            a["geometry"]=g
            rows.append(a)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")

geom_cache = data_dir / "highlands_ranch_geometry_20260401.parquet"
if geom_cache.exists():
    parcel_geom = gpd.read_parquet(geom_cache)
else:
    spns = attrs["PARCEL"].dropna().astype(str).unique().tolist()
    try:
        parcel_geom = fetch_geometries(spns, parcel_query_url)
    except Exception:
        parcel_geom = fetch_geometries(spns, fallback_parcel_query_url)
    parcel_geom.to_parquet(geom_cache, index=False)

gdf = gpd.GeoDataFrame(parcel_geom.merge(attrs, left_on="PARCEL_SPN", right_on="PARCEL", how="inner"), geometry="geometry", crs="EPSG:4326")
for col in ["LANDACT","IMPACT","TOTALACT","LANDASD","IMPASD","TOTALASD","SALEP","GIS_Acres"]:
    gdf[col]=pd.to_numeric(gdf.get(col), errors="coerce").fillna(0)
gdf["is_osm_surface_parking"] = False
print(gdf.shape)


## Step 3: Stamp zoning


In [ ]:
def load_zoning():
    cache = data_dir / "douglas_zoning_20260401.parquet"
    if cache.exists():
        return gpd.read_parquet(cache)
    session = requests.Session()
    count = session.get(zoning_query_url, params={"f":"json","where":"1=1","returnCountOnly":"true"}, timeout=60).json().get("count",0)
    feats=[]
    for off in range(0, int(count), 2000):
        r = session.get(zoning_query_url, params={
            "f":"json","where":"1=1",
            "outFields":"OBJECTID,ZONE_TYPE,FIRST_DESC",
            "returnGeometry":"true","resultOffset":off,"resultRecordCount":2000,"outSR":4326,
        }, timeout=180)
        r.raise_for_status(); feats.extend(r.json().get("features",[]))
    rows=[]
    for f in feats:
        a=f.get("attributes",{})
        g=esri_poly_to_shapely(f.get("geometry",{}))
        if g is not None:
            a["geometry"]=g; rows.append(a)
    zgdf=gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    zgdf.to_parquet(cache, index=False)
    return zgdf

zoning_gdf = load_zoning()
gdf_3857 = gdf.to_crs(epsg=3857)
pts = gpd.GeoDataFrame(gdf_3857.drop(columns=["geometry"]), geometry=gdf_3857.geometry.centroid, crs="EPSG:3857")
stamped = gpd.sjoin(pts, zoning_gdf.to_crs(epsg=3857)[["ZONE_TYPE","FIRST_DESC","geometry"]], how="left", predicate="within")
gdf["ZONING_CLASS"] = gdf["PARCEL"].map(stamped.groupby("PARCEL")["ZONE_TYPE"].first())
gdf["ZONING_DESC"] = gdf["PARCEL"].map(stamped.groupby("PARCEL")["FIRST_DESC"].first())


## Step 4: Categories + modeling


In [ ]:
gdf["IR"] = np.where(gdf["TOTALASD"] > 0, gdf["IMPASD"] / gdf["TOTALASD"], 0)
gdf["is_vacant"] = gdf["Vacant_Flag"].fillna("").str.upper().eq("Y")

def map_property_category(account_type, valuation_desc):
    code = str(account_type).upper(); desc = str(valuation_desc).upper()
    if code == "V" or "VAC" in desc: return "Vacant / Undeveloped"
    if code == "R" or any(x in desc for x in ["SINGLE","CONDO","TOWN","APART"]): return "Residential"
    if code == "C" or any(x in desc for x in ["RETAIL","OFFICE","COMMERCIAL"]): return "Commercial"
    if code == "I" or "INDUSTR" in desc: return "Industrial"
    if code == "A" or "AGRIC" in desc: return "Agricultural"
    if code == "X": return "Exempt / Government"
    return "Other"

gdf["PROPERTY_CATEGORY"] = gdf.apply(lambda r: map_property_category(r.get("ACCTTYPE"), r.get("Valuation_Description")), axis=1)
gdf["ANALYSIS_CATEGORY"] = np.where(gdf["ZONING_CLASS"].fillna("").eq("PD") & gdf["PROPERTY_CATEGORY"].eq("Residential"), "Residential - PD", gdf["PROPERTY_CATEGORY"])
analysis_gdf = gdf[(gdf["PROPERTY_CATEGORY"] != "Exempt / Government") & (gdf["PROPERTY_CATEGORY"] != "Agricultural")].copy()

model_df = analysis_gdf.copy()
model_df["current_tax"] = model_df["TOTALASD"] / 1000.0
current_revenue = model_df["current_tax"].sum()

scenario_outputs = {}
for ratio in [2,4,8]:
    land_mill, imp_mill, revenue, scenario_df = model_split_rate_tax(
        df=model_df.copy(),
        land_value_col="LANDASD",
        improvement_value_col="IMPASD",
        current_revenue=current_revenue,
        land_improvement_ratio=ratio,
    )
    scenario_df["tax_change"] = scenario_df["new_tax"] - scenario_df["current_tax"]
    scenario_df["tax_change_pct"] = np.where(scenario_df["current_tax"] > 0, (scenario_df["tax_change"] / scenario_df["current_tax"]) * 100, 0)
    scenario_outputs[ratio] = {"land_mill": land_mill, "imp_mill": imp_mill, "revenue": revenue, "df": scenario_df}
    print(f"{ratio}:1 split -> land mill {land_mill:.4f}, imp mill {imp_mill:.4f}, revenue {revenue:,.2f}")


## Step 5: Summary, diagnostics, and equity


In [ ]:
primary = scenario_outputs[4]["df"].copy()

summary = calculate_category_tax_summary(primary, "ANALYSIS_CATEGORY", "current_tax", "new_tax")
print_category_tax_summary(summary)

analysis_df = primary.copy()
analysis_df["LAND_USE_FOR_ANALYSIS"] = np.select([
    analysis_df["is_vacant"] == True,
    analysis_df["is_osm_surface_parking"] == True,
], ["Vacant Land", "Trans - Parking"], default="Other")

vacant_results = analyze_vacant_land(analysis_df, "LANDASD", "IMPASD", "LAND_USE_FOR_ANALYSIS", vacant_identifier="Vacant Land", owner_col="NAME")
print_vacant_land_summary(vacant_results)
parking_results = analyze_parking_lots(analysis_df, "LANDASD", "IMPASD", "LAND_USE_FOR_ANALYSIS", parking_identifier="Trans - Parking", min_land_value_threshold=50000, max_improvement_ratio=0.10)
print_parking_analysis_summary(parking_results)
display(pd.DataFrame(analyze_land_by_improvement_share(analysis_df, "LANDASD", "IMPASD")["categories"]))

try:
    _, census_boundaries = get_census_data_with_boundaries(fips_code="08035", year=2022)
    matched = match_to_census_blockgroups(primary.to_crs(epsg=4326), census_boundaries, join_type="left")
    matched = matched[matched["median_income"] > 0].copy()
    grouped = matched.groupby("std_geoid").agg(
        median_income=("median_income", "first"),
        minority_pct=("minority_pct", "first"),
        total_current_tax=("current_tax", "sum"),
        total_new_tax=("new_tax", "sum"),
        mean_tax_change=("tax_change", "mean"),
    ).reset_index()
    grouped["mean_tax_change_pct"] = np.where(grouped["total_current_tax"] > 0, ((grouped["total_new_tax"] - grouped["total_current_tax"]) / grouped["total_current_tax"]) * 100, 0)
    display(create_quintile_summary(grouped, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct"))
    display(create_quintile_summary(grouped, "minority_pct", "minority_pct", "mean_tax_change", "mean_tax_change_pct"))
except Exception as exc:
    print(f"Census section skipped: {exc}")
